# Phase 4 — The agent as an experimental subject

The Phase-3 agent makes a **choice**: when the retrieved context looks thin, it decides to
re-query. This notebook measures the causal effect *of that choice* — not on fixed inputs, but on
the model's own endogenous behavior. That's the hard, under-served kind of evaluation: you're
scoring a **policy**, not a function.

The move: an agent's decision to re-query is like a subject deciding whether to take a pill. We
assign the *option* to re-query; the model chooses *uptake*. Two estimands fall out:

- **ITT (intent-to-treat)** — the effect of *giving the agent the option*, averaged over **all**
  questions whether they used it or not. The ship-it number.
- **CACE (complier average causal effect)** — the effect of re-querying among the **compliers**:
  the questions where the agent actually chose to take a second hop.

They're linked: `ITT = P(re-query) × CACE`. ITT is CACE diluted by every question that never used
the capability.

### Why a crossover (and why that's a luxury here)
With human subjects you can't un-treat a person, so you'd need instrumental variables to back out
the complier effect. A **model has no memory across API calls**, so we can run the *same question*
through both arms and observe its counterfactual directly — then read the complier effect off by
subsetting, no IV needed. That's the payoff of the subject being a model.

> **Runtime:** Colab, **CPU is fine** (no local LLM). Secret: `ANTHROPIC_API_KEY`. This makes many
> Claude calls (~a few hundred) — budget roughly $1–3; turn `N_QUESTIONS`/`R` down to spend less.

## Setup

In [1]:
!pip install -qU langchain langchain-community langchain-huggingface langchain-qdrant
!pip install -qU qdrant-client sentence-transformers beautifulsoup4 lxml anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 44.3 MB/s eta 0:00:00


In [3]:
import os, re, time, textwrap
import requests
import numpy as np
from bs4 import BeautifulSoup
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')

import anthropic
from pydantic import BaseModel
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from sentence_transformers import CrossEncoder

SEC_USER_AGENT = "David Schaaf davidschaaf@berkeley.edu"   # edit to your name/email
AGENT_MODEL = "claude-opus-4-8"
MAX_STEPS = 5
client = anthropic.Anthropic()
print("Setup ready.")

Setup ready.


## Rebuild retrieval + the agent's tool *(provided — Phases 1–3)*

In [ ]:
COMPANIES = {"AAPL": 320193, "MSFT": 789019, "NVDA": 1045810}

def fetch_latest_10k_html(ticker, cik):
    h = {"User-Agent": SEC_USER_AGENT}
    recent = requests.get(f"https://data.sec.gov/submissions/CIK{cik:010d}.json",
                          headers=h).json()["filings"]["recent"]
    i = next(j for j, f in enumerate(recent["form"]) if f == "10-K")
    acc, doc = recent["accessionNumber"][i].replace("-", ""), recent["primaryDocument"][i]
    html = requests.get(f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc}/{doc}",
                        headers=h).text
    time.sleep(0.5)
    return html

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
documents = []
for tk, cik in COMPANIES.items():
    print("Fetching", tk, "...")
    soup = BeautifulSoup(fetch_latest_10k_html(tk, cik), "lxml")
    for t in soup(["script", "style"]):
        t.decompose()
    for chunk in splitter.split_text(re.sub(r"\s+", " ", soup.get_text(" ")).strip()):
        documents.append(Document(page_content=chunk,
            metadata={"chunk_id": len(documents), "ticker": tk}))

embeddings = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")
qdrant = QdrantVectorStore.from_documents(
    documents, embeddings, location=":memory:", collection_name="sec_causal")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve_reranked(query, fetch_k=10, top_k=4):
    cands = qdrant.similarity_search(query, k=fetch_k)
    scores = reranker.predict([(query, d.page_content) for d in cands])
    return [d for _, d in sorted(zip(scores, cands), key=lambda x: x[0], reverse=True)][:top_k]

def do_search(query, k=4):
    docs = retrieve_reranked(query, top_k=k)
    ids = [d.metadata["chunk_id"] for d in docs]
    text = "\n\n".join(f"[id={d.metadata['chunk_id']} | {d.metadata['ticker']}] {d.page_content}"
                       for d in docs)
    return text, ids

SEARCH_TOOL = {
    "name": "search_filings",
    "description": ("Search the SEC 10-K filings for passages relevant to a query. If the results "
                    "don't fully answer the question, search again with a refined query."),
    "input_schema": {"type": "object",
                     "properties": {"query": {"type": "string", "description": "What to search for"}},
                     "required": ["query"]},
}
SYSTEM = ("You answer questions about SEC 10-K filings for Apple, Microsoft, and NVIDIA using the "
          "search_filings tool. If the passages don't fully answer the question, refine your query "
          "and search again. Answer only from retrieved passages, cite ids like [42], and if the "
          "filings don't contain the answer, say so. Be concise.")
print(f"Retrieval + tool ready over {len(documents)} chunks.")

---
## 1 · One agent, one knob: a hop cap

The whole experiment hinges on changing **exactly one thing** — whether the agent may take a
second retrieval. The clean way is a single agent function with a `max_hops` cap:
- **Control** = `max_hops=1` — the agent does its (possibly reformulated) first search, then is
  forced to answer.
- **Treatment** = `max_hops=MAX_STEPS` — free to re-query.

Because both arms share the same first-hop behavior, the only difference is the extra hop —
so a question that *wouldn't* re-query anyway (a never-taker) gets **identical** treatment in both
arms. That's what keeps the comparison clean.

In [25]:
# --- YOU WRITE: the agent loop, generalized with a hop cap ---
# Same ReAct loop as Phase 3, but only offer SEARCH_TOOL while len(searches) < max_hops.
# Once the cap is hit, call the model with NO tools so it must answer from what it has.
# Track total tokens (sum resp.usage.input_tokens + output_tokens across calls).
# Return {"answer", "hops": len(searches), "tokens"}.
def run_agent_capped(question, max_hops):
    ### YOUR CODE HERE
    question_prompt = {"role": "user", "content": question}
    messages = [question_prompt]
    searches = []
    tokens = 0

    # SEARCH_TOOL available
    for _ in range(MAX_STEPS):
        kw = dict(model=AGENT_MODEL, max_tokens=1024, system=SYSTEM,
                  messages=messages)
        if len(searches) < max_hops:
            kw["tools"] = [SEARCH_TOOL]

        response = client.messages.create(**kw)
        tokens += response.usage.input_tokens + response.usage.output_tokens
        if response.stop_reason != "tool_use":
            break

        # Only if we need to use the tool search_filings
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                query = block.input.get("query", "")
                searches.append(query)
                text, _ = do_search(**block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": text})
        messages.append({"role": "user", "content": tool_results})

    if response.stop_reason == "tool_use":
        kw = dict(model=AGENT_MODEL, max_tokens=1024, system=SYSTEM,
                  messages=messages)
        response = client.messages.create(**kw)
        tokens += response.usage.input_tokens + response.usage.output_tokens

    answer = "".join(b.text for b in response.content if b.type == "text")
    return {"answer": answer, "hops": len(searches), "tokens": tokens}

print("Agent ready.")

Agent ready.


In [16]:
# --- INSTRUMENT: same question, both caps — see the behavior diverge ---
q = "What are NVIDIA's main risk factors, and how does it describe competition in its industry?"
c = run_agent_capped(q, max_hops=1)
t = run_agent_capped(q, max_hops=MAX_STEPS)
print(f"control  hops={c['hops']}  tokens={c['tokens']}")
print(f"treatment hops={t['hops']}  tokens={t['tokens']}")
print("\nTreatment answer:\n", textwrap.fill(t["answer"], 100))

Message(id='msg_015yKwNURpFt1GPdCjviWPvi', container=None, content=[TextBlock(citations=None, text="I'll search for NVIDIA's risk factors and competition disclosures.", type='text'), ToolUseBlock(id='toolu_0193FMXH9outaPcoPDWmwvxV', caller=DirectCaller(type='direct'), input={'query': 'NVIDIA main risk factors'}, name='search_filings', type='tool_use'), ToolUseBlock(id='toolu_013VoysQ7N6LYAsPb9xTmjFh', caller=DirectCaller(type='direct'), input={'query': 'NVIDIA competition in industry competitors'}, name='search_filings', type='tool_use')], model='claude-opus-4-8', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=542, output_tokens=141, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard'))
[{'

## The outcome — a blind completeness judge *(provided)*

Re-querying should improve **completeness** (a one-hop answer can be perfectly faithful to thin
context yet miss half the question). A Claude judge scores each answer 1–5 on how fully it
addresses the question. It never sees which arm produced the answer, so it's **blind by
construction** — the single most important guard against biasing the outcome.

In [17]:
class Completeness(BaseModel):
    score: int   # 1 (misses most of the question) .. 5 (fully addresses every part)
    reason: str

def judge_completeness(question, answer):
    prompt = ("Score how COMPLETELY the answer addresses the question, 1-5 "
              "(1=misses most, 5=addresses every part). Judge completeness only.\n\n"
              f"Question: {question}\n\nAnswer: {answer}")
    return client.messages.parse(model=AGENT_MODEL, max_tokens=300,
        messages=[{"role": "user", "content": prompt}], output_format=Completeness).parsed_output

print("Blind completeness judge ready.")

Blind completeness judge ready.


## The question set *(provided)*

Deliberately weighted toward **compound / multi-part** questions — those are the ones that give the
agent a reason to re-query (compliers). A set of simple single-fact questions would be all
never-takers, with no effect to measure. A few simple ones are included so the never-taker case is
represented too.

In [18]:
QUESTIONS = [
    "What are NVIDIA's main risk factors, and how does it describe competition?",
    "What does Apple say about its revenue, and what risks does it tie to supply chain?",
    "How does Microsoft describe its cloud business and the competition it faces there?",
    "What does NVIDIA say about data-center demand and the risks to that demand?",
    "Summarize Apple's regulatory risks and its risks related to foreign operations.",
    "How does Microsoft describe both its AI investments and the risks around them?",
    "What are Apple's main product risks, and what does it say about R&D spending?",
    "What does NVIDIA say about its supply concentration and about intellectual property?",
    "How does Microsoft describe its competition, and what does it say about cybersecurity risk?",
    "What are the main risk factors Apple highlights?",            # simpler
    "How does NVIDIA describe its gaming segment?",                # simpler
    "What does Microsoft say about its revenue sources?",          # simpler
]
N_QUESTIONS = len(QUESTIONS)
R = 3   # independent runs per question per arm
print(f"{N_QUESTIONS} questions x {R} runs x 2 arms = {N_QUESTIONS*R*2} agent trajectories.")

12 questions x 3 runs x 2 arms = 72 agent trajectories.


---
## 2 · Run the crossover experiment

For each question, `R` independent replicates. Each replicate runs **both** arms and judges both
answers, so we observe the counterfactual per replicate. We log, per run:
`Y_control`, `Y_treated`, `D` (did the treatment arm re-query — `hops >= 2`), and tokens.

*(This is the slow cell — it makes all the Claude calls.)*

In [26]:
# --- YOU WRITE: the experiment loop ---
# For each question, for r in range(R):
#   ctrl = run_agent_capped(q, 1); trt = run_agent_capped(q, MAX_STEPS)
#   append a record with q_idx, run, Y_control (judge ctrl), Y_treated (judge trt),
#   D = int(trt hops >= 2), hops, tokens (ctrl+trt).
def run_experiment(questions, repeats):
    ### YOUR CODE HERE
    output = []
    for question_idx, question in enumerate(questions):
        for r in range(repeats):
            ctrl = run_agent_capped(question, 1)
            trt = run_agent_capped(question, MAX_STEPS)
            output.append({
                "q_idx": question_idx,
                "run": r,
                "Y_control": judge_completeness(question, ctrl["answer"]).score,
                "Y_treated": judge_completeness(question, trt["answer"]).score,
                "D": int(trt["hops"] >= 2),
                "hops": trt["hops"],
                "tokens": ctrl["tokens"] + trt["tokens"]
            })
        print(f"  done q{question_idx+1}/{len(questions)}")
    return output



In [ ]:
# RUN EXPERIMENT
records = run_experiment(QUESTIONS, R)
print(f"\nlogged {len(records)} runs")

In [ ]:
# --- INSTRUMENT: peek at the raw log ---
print(f"{'q':>3}{'run':>4}{'Yc':>4}{'Yt':>4}{'D':>3}{'hops':>5}")
for rec in records[:8]:
    print(f"{rec['q_idx']:>3}{rec['run']:>4}{rec['Y_control']:>4}{rec['Y_treated']:>4}"
          f"{rec['D']:>3}{rec['hops']:>5}")
print(f"\noverall re-query rate: {np.mean([r['D'] for r in records]):.2f}")

---
## 3 · The estimands — ITT, P(re-query), CACE

Point estimates use **every run** (this is where the repeated runs legitimately help — they average
down the model's run-to-run noise):
- **ITT** = mean of `Y_treated − Y_control` over all runs.
- **P(re-query)** = mean of `D`.
- **CACE** = mean of `Y_treated − Y_control` over just the runs where `D == 1`. Because each of
  those is differenced against **its own** control run, this within-run comparison is unconfounded —
  no need for IV here.

You should see `ITT ≈ P(re-query) × CACE`.

In [28]:
# --- YOU WRITE: compute the three estimands ---
# diffs = Y_treated - Y_control per run; D = the compliance flags.
# ITT = mean(diffs); p_requery = mean(D); CACE = mean(diffs where D==1).
### YOUR CODE HERE

diffs = np.array([r["Y_treated"] - r["Y_control"] for r in records], float)
D = np.array([r["D"] for r in records])

ITT = np.mean(diffs)
p_requery = np.mean(D)
CACE = diffs[D == 1].mean() if D.any() else float("nan")

print(f"ITT (ship-it effect):        {ITT:+.3f}")
print(f"P(re-query):                 {p_requery:.3f}")
print(f"CACE (effect on compliers):  {CACE:+.3f}")
print(f"check  P(re-query) x CACE =  {p_requery * CACE:+.3f}   (should ~= ITT)")

NameError: name 'records' is not defined

---
## 4 · Inference — respect the clustering *(the part people get wrong)*

The `R` runs of one question are **not** independent observations — they're repeated measures on the
same item, so they're correlated. Pouring all `N × R` runs into a permutation test or CI as if
independent is **pseudoreplication**: it fakes 3× the information and makes everything look more
significant than it is. The independent unit is the **question**.

So point estimates used every run (above), but **inference resamples questions, not runs**:
- **ITT p-value** — reduce each question to its mean effect `d_q`, then a **sign-flip permutation
  test** over the `N` questions (under the null, each `d_q`'s sign is a coin flip).
- **CIs** — **cluster bootstrap**: resample *questions* with replacement (each brings all its runs),
  recompute the estimand each time, take the 2.5/97.5 percentiles.

In [ ]:
# --- YOU WRITE: question-level inference ---
# by_q: group records by q_idx.
# (a) d_q = per-question mean of (Y_treated - Y_control); observed ITT = mean(d_q).
#     sign-flip permutation: many times, multiply each d_q by a random +/-1, take the mean;
#     p = fraction of |null means| >= |observed|.
# (b) cluster bootstrap: resample QUESTIONS with replacement, gather their runs, recompute
#     ITT and CACE; take 2.5/97.5 percentiles. Drop bootstrap samples with no compliers for CACE.
rng = np.random.default_rng(0)
### YOUR CODE HERE
by_q = {q_idx: [r for r in records if r["q_idx"] == q_idx] for q_idx in range(N_QUESTIONS)}

# a - randomization inference
d_q = np.mean(np.array([r["Y_treated"] - r["Y_control"] for r in recs])) for recs in by_q.values()
obs = np.mean(d_q)
null = np.array([(rng.choice([-1, 1], size=len(d_q)) * d_q).mean() for _ in range(5000)])
p_itt = np.mean(np.abs(null) >= abs(obs))

# b - clustering
def cluster_bootstrap_ci(estimand_fn, n_boot=3000):
    question_clusters = list(by_q.values())        # each cluster = one question's list of runs
    n_questions = len(question_clusters)
    bootstrap_estimates = []

    for _ in range(n_boot):
        # resample QUESTIONS (clusters) with replacement, then pull in all of their runs
        sampled_question_indices = rng.choice(n_questions, size=n_questions)
        resampled_runs = [
            run
            for question_index in sampled_question_indices
            for run in question_clusters[question_index]
        ]

        estimate = estimand_fn(resampled_runs)
        if not np.isnan(estimate):                 # skip resamples where the estimand is undefined
            bootstrap_estimates.append(estimate)

    return np.percentile(bootstrap_estimates, [2.5, 97.5])


def itt_estimand(runs):
    return np.mean([run["Y_treated"] - run["Y_control"] for run in runs])


def cace_estimand(runs):
    complier_runs = [run for run in runs if run["D"] == 1]
    if not complier_runs:                          # no compliers in this resample → undefined
        return float("nan")
    return np.mean([run["Y_treated"] - run["Y_control"] for run in complier_runs])



print(f"ITT  {ITT:+.3f}   95% CI [{itt_ci[0]:+.3f}, {itt_ci[1]:+.3f}]   perm p = {p_itt:.3f}")
print(f"CACE {CACE:+.3f}   95% CI [{cace_ci[0]:+.3f}, {cace_ci[1]:+.3f}]")

**🎤 Talk-track — What this buys you.** *"I treated the agent as an experimental subject and
measured the effect of a decision it makes on its own. ITT is what shipping self-correction does to
average completeness; CACE is what it does on the questions that actually trigger it — and CACE is
the bigger, mechanism-specific number because ITT is diluted by every question that never re-queried.
The one thing I was careful about: the repeated runs cut noise but don't multiply my sample size —
the unit of inference is the question, so I permute and bootstrap questions, not runs. That
distinction is the difference between a real result and a pseudoreplicated one."*

---
## Recap — and the honest caveats

You measured the causal effect of the agent's **own choice**, with two estimands (ITT vs CACE), a
blind outcome, and inference that respects the clustering. That's model/behavior evaluation, not
input evaluation — the thing teams struggle with once they ship agents.

**Where this is fragile (say these before someone else does):**
- **Outcome validity.** Completeness is judge-scored; a blind, paired design controls a lot, but the
  judge is itself a model. Spot-check it against a few of your own labels before trusting the number.
- **Exclusion assumption.** CACE-by-subsetting is clean *only if* never-takers get identical
  treatment in both arms. They do here because both arms share the first hop — but change the arms
  and you'd have to re-argue it.
- **Stochastic compliance.** `D` is a per-run coin flip near the margin; runs reduce its noise but a
  borderline question can land on either side.
- **External validity → this is where IV comes back.** In production you can't re-run the same query
  under both configs, so you lose the crossover and the direct CACE. There you'd randomize
  *availability* and recover CACE with instrumental variables (`ITT_Y / ITT_D`) — same estimand,
  harder identification.